# Mecanismo de Atenção
## (A trilha acadêmica)

### 1. Introdução
Os notebooks AIAYN, ViT e Swin visam ensinar o básico sobre os transformers e como são aplicados à visão computacional. É de grande importância entender a atenção por ser a revolução que possibilitou o surgimento das IAs generativas.

## 2. Attention Is All You Need

<div align="center">
    <img src="https://shreyansh26.github.io/assets/img/posts_images/attention/featured.png" width="800">
</div>
https://shreyansh26.github.io/assets/img/posts_images/attention/featured.png

Em 2017 é lançado no Arxiv, por uma equipe do Google, o artigo entitulado "Attention Is All You Need", que introduz o mecanismo de Self-Attention e revoluciona a computação, dando o arcabouço para as IAs Generativas atuais. O artigo pretende alcançar um novo estado da arte para a tradução, mas a teoria será posteriormente usada para os Transformers.

## 3. Entendendo Atenção

O artigo contém uma imagem para ajudar em seu entendimento. Nessa primeira parte exploraremos cada passo se guiando pela imagem.

<div align="center">
    <img src="https://miro.medium.com/1*f3L_gGaNy9wVuTenyQJLEA.png" width="300">
</div>
https://miro.medium.com/1*f3L_gGaNy9wVuTenyQJLEA.png

#### 3.1 Input

##### 3.1.1 Token e Tokenização

Primeiramente, precisamos entender o que é um token. Para isso, vamos supor que você e seus amigos estão em um campeonato de League of Legends. Vocês combinaram táticas previamente, mas, no meio da partida, não conseguem comunicá-las em um momento decisivo. Para contornar esse problema, vocês associam cada tática a um número. Por exemplo, um dos membros pode gritar “Tática 1”, e todos entendem que devem focar no Barão, pois já foi previamente definido que o número 1 se refere a essa ação.

Os tokens funcionam de forma semelhante: eles representam informações por meio de identificadores numéricos. Por exemplo, poderíamos associar 1 a “cachorro”, 2 a “gato”, 3 a “Zweihander”, e assim por diante. Esse processo de converter elementos de linguagem em unidades numéricas é chamado de tokenização.

Com base nessas ideias, podemos entender como isso é implementado na prática. O artigo propõe o uso de vetores de 512 dimensões, ou seja, cada token é representado por uma sequência de 512 números. Inicialmente, esses vetores são atribuídos de forma aleatória, mas são ajustados durante o treinamento do modelo. Isso pode parecer contraintuitivo à primeira vista: por que treinar a representação de uma palavra? Considere, por exemplo, as palavras “rei” e “rainha” em comparação com “rei” e “matemática”. Intuitivamente, percebemos que o primeiro par possui maior relação semântica. Durante o treinamento, o modelo ajusta os vetores de modo que “rei” e “rainha” fiquem mais próximos entre si no espaço vetorial do que “rei” e “matemática”.

Além disso, em modelos bem treinados, relações semânticas mais abstratas também emergem. Por exemplo, a diferença vetorial entre “homem” e “mulher” tende a ser semelhante à diferença entre “rei” e “rainha”, refletindo padrões consistentes de analogia no espaço de embeddings.

Complemento: Para fins didáticos, utilizamos uma simplificação. Na prática, tokens nem sempre correspondem a palavras inteiras. Muitas vezes, são fragmentos, como prefixos, sufixos ou radicais. Por exemplo, a palavra “injusto” pode ser decomposta em dois tokens: “in” e “justo”. Em modelos modernos, essa decomposição pode ser ainda mais granular. Ainda assim, a ideia central permanece: cada token corresponde a uma unidade de informação representada numericamente.

##### 3.1.2 Input Embedding

Considere agora para prosseguirmos um exemplo. O modelo recebeu como input, "O sombrio Imu aparece diante do herói Luffy". Primeiramente, o modelo traduzirá isso em tokens. Ele pode observar sombrio e vai buscar no seu dicionário de tokens, encontrando um vetor que será usado a partir daqui para representar essa palavra.

#### 3.2 Positional Encoding

Porém, imaginemos agora a frase: "Felipe Alves, Felipe Moura and Augusto Alves praised the sun" \\_/. O modelo ao passar pela tokenização, atribui o mesmo vetor para ambas as palavras Felipe. Dessa forma, é impossível de o modelo distinguir palavras iguais, o que atrapalha significativamente o entendimento da frase pelo modelo. Aparece então, uma brilhante ideia, vital ao Self-Attention, chamada Positional Encoding. Como o próprio nome diz, é sobre dar uma representação a cada posição. Isso é feita de maneira simples, basta atribuir um novo vetor (de 512 dimensões nesse caso) para representar a posição. Esse vetor não é atualizado, ou seja, para todas as frases recebidas, o vetor de posição da primeira palavra será sempre igual (e o mesmo vale para as demais). Assim, o modelo aprenderá sobre a posição de cada token na frase por meio desse acréscimo.
No artigo, é proposto um vetor de positional encoding, com pos a posição do token e i a dimensão (posição no vetor):
$$
PE(pos, 2i) = sin(\frac{pos}{10000})^{\frac{2i}{512}}
$$
$$
PE(pos, 2i+1) = cos(\frac{pos}{10000})^{\frac{2i}{512}}
$$
Dessa forma, palavras iguais serão distinguíveis entre si.

:O Complemento: Esse é só um tipo de Positional Encoding. Uma mais básica usada é atribuir um vetor aleatório para cada posição não treinável e, no BERT, é usada essa ideia, mas com um vetor treinável. Um exemplo de PE muito mais usado na atualidade é o Relative Positional Encoding. Nesse,é usada a distância entre pares de palavras. Seja uma imagem 3x3, por exemplo, um RPE poderia ser:

\begin{bmatrix}
    (-1,-1) & (0,-1) & (1,-1) \\
    (-1,0) & (0,0) & (1,0) \\
    (-1,1) & (0,1) & (1,1)
\end{bmatrix}


### 3.3 Attention Head

Agora, finalmente chegamos a uma das partes mais cruciais para entender a atenção. Nós precisamos entender as conexões entre cada token. Vou começar com uma analogia. Duas pessoas estão analisando uma frase, a primeira pergunta sobre se uma palavra tem correlação com alguma outra e a segunda responde para cada palavra. 

"O sombrio Imu aparece diante do herói Luffy"

A primeira pessoa pergunta "Luffy", a segunda começa "O" pouca, "sombrio" pouca, ..., "herói" muita, "Luffy" média.

Atribuindo valores de 0 a 1 normalizados para essas respostas, ficaria parecido com [0, 0, 0, 0, 0, 0, 0.9, 0.1].

Se isso fez sentido, você já entendeu a primeira parte da cabeça de atenção. Na analogia, a pessoa que pergunta seria o vetor Q (query), enquanto a que responde K (key), com a resposta dada pelo produto escalar $QK^T$.

Veja como ficaria em uma frase completa com exemplo do 3Blue1Brown:
<div align="center">
    <img src="https://miro.medium.com/v2/resize:fit:1400/0*-O-xCeusDPqDKzt-.png" width="800">
</div>
https://miro.medium.com/v2/resize:fit:1400/0*-O-xCeusDPqDKzt-.png

Tendo a correlação entre pares de tokens, temos por fim o V (Value), que representa agora a informação. Ou seja, V dá um peso para cada correlação. 

Mas de onde vem cada um desses 3 elementos? A cabeça de atenção recebe um vetor de input. Esse vetor passará por uma tranformação linear, sendo multiplicado por $W^Q$, $W^K$ e $W^V$ para obter o vetor Q, K e V reespectivamente. Esses vetores de peso são treinados no modelo para exercerem seus papéis. O motivo de esses vetores serem resultado de multiplicação escalar somente é que se tivesse um bias, ele introduziria ruído, pois não faria sentido para obter similaridade entre as entradas. Após ter cada um desses vetores, o output da Cabeça de Atenção será dado por:
$$
Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

### 3.5 Multi-Head Attention

Como o próprio nome diz, a Multi-Head é a junção de mais de uma cabeça de atenção. Isso é feito de maneira intuitiva:
$$
MultiHead(Q,K,V) = Concat(head_1, head_2, ..., head_h) W^O
$$
Com $W^O \in \mathbb{R}^{hd_v\times d_{model}}$ e $d_v = d_{model}/h$ sendo também treinável.

No artigo, cada Multi-Head era composta de 8 cabeças (h=8). O interesse em fazer várias cabeças é de que cada uma pode buscar características diferentes, tendo uma maior gama de possibilidades para o modelo.

Uma variação importante é a Masked Multi-Head Attention, em que é como K respondendo Q somente quanto ao que veio antes dele. Ou seja:
$$
Q_iK_j = - \infty, \space se \space j > i
$$

### 3.6 Add & Norm

Após cada sub-layer, é feito a conexão residual, mecanismo de estabilização para evitar vanishing e exploding gradients.:
$$
output = Layernorm(x + Sublayer(x))
$$
O Layernorm faz o vetor ter elementos com aproximadamente média 0 e variância 1.

### 3.7 Feed Foward

É uma transformação linear seguida de ReLU e nova transformação linear.
$$
FFN(x) = max(0, xW_1 + b_1)W_2 + b_2
$$

### 3.8 Arquitetura

Agora que entendemos cada caixinha da figura, vamos entender o conjunto.

<div align="center">
    <img src="https://miro.medium.com/1*f3L_gGaNy9wVuTenyQJLEA.png" width="300">
</div>
https://miro.medium.com/1*f3L_gGaNy9wVuTenyQJLEA.png

Primeiramente, recebemos um input, que passa pelo Embedding e Positional Encoding. Em seguida, passará por um agrupamento de passos N vezes (no caso do artigo N = 6). Em cada um, primeiramente passa por uma Multi-Head Attention, depois por Add & Norm, por Feef Forward e por fim Add & Norm.

Agora, o Ouput shifted right, ou seja, cada token indo para a posição depois (o primeiro agora é o segundo e assim por diante) passa por Embedding e Positional Encoding. Agora novamente repetido 6 vezes, esse output passa por uma Masked Multi-Head Attention, Add & Norm, e agora temos uma Multi-Head Attention em que Q vem do output, enquanto K e V do input. Depois passa por Add & Norm com residual do output, Feed Forward e Add& Norm.

Por fim, passa por uma transformação linear, um softmax e obtém o resultado. Esse resultado é dado como probabilidade de ser cada palavra. Por exemplo, 90% de ser tijolo, 2% de ser madeira,... em que o modelo escreveria tijolo.

## 4. Modelos pré-treinados

Os Transformers possuem milhões, em muitos casos, centenas de bilhões, de parâmetros a serem treinados, o que torna seu treinamento extremamente custoso em termos computacionais. Para aplicações acadêmicas, por exemplo, frequentemente é inviável treinar um modelo desse porte do zero, devido à necessidade de grandes volumes de dados e alto poder computacional.

Diante disso, surgem os modelos pré-treinados, isto é, redes neurais que já foram treinadas previamente em grandes volumes de texto, aprendendo padrões gerais da linguagem. Esse conhecimento adquirido é, então, reutilizado em tarefas específicas, permitindo que os usuários partam de um modelo com parâmetros já bem ajustados, em vez de iniciar o treinamento do zero.

O processo de adaptar um modelo pré-treinado para uma tarefa específica é chamado de fine-tuning. Nesse contexto, o modelo passa por um treinamento adicional em um dataset mais específico, ajustando seus parâmetros de forma mais refinada para se adequar ao problema em questão.

A partir dessa abordagem, destacam-se dois dos principais modelos baseados em Transformers: o BERT e o GPT.

O BERT utiliza apenas o encoder do Transformer e é treinado de forma bidirecional, ou seja, levando em consideração tanto o contexto à esquerda quanto à direita de cada palavra. Seu pré-treinamento é baseado principalmente na tarefa de Masked Language Modeling, na qual algumas palavras da frase são mascaradas e o modelo precisa prevê-las com base no restante do contexto. Essa abordagem faz com que o BERT seja especialmente eficaz em tarefas que exigem compreensão profunda de texto, como classificação, análise de sentimento e resposta a perguntas.

Por outro lado, o GPT utiliza apenas o decoder do Transformer e é treinado de forma auto-regressiva, prevendo o próximo token com base apenas nos anteriores. Isso significa que o modelo aprende a gerar texto de maneira sequencial, construindo frases palavra por palavra. Essa característica o torna particularmente eficiente em tarefas de geração de texto, como escrita automática, chatbots e geração de código.

Por fim, é importante destacar que o modelo implementado neste notebook segue a arquitetura básica de um Transformer, mas é treinado do zero para uma tarefa específica. Em contraste, modelos como BERT e GPT utilizam pré-treinamento em larga escala, o que lhes permite capturar padrões gerais da linguagem e, posteriormente, serem adaptados com eficiência para diferentes aplicações.

## 5. Aplicação de Transformers

Para testarmos o Transformers, vejamos uma aplicação simples. Usaremos um dataset de DRX para a predição de estrutura cristalina.

Primeiro importamos as bibliotecas e carregamos nosso dataset

In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

df = pd.read_csv('xrd_dataset_full.csv.gz', compression='gzip')
print(df.shape)
print(df['crystal_system'].value_counts())

(91145, 9)
crystal_system
orthorhombic    27352
tetragonal      16332
cubic           14732
trigonal        12172
hexagonal       10175
triclinic        8850
monoclinic       1532
Name: count, dtype: int64


Agora, transformamos os picos do espectro em um vetor que simboliza isso, pois o modelo consegue aprender melhor assim. Por exemplo, se o pico fosse na posição 3, em vez de enviar 3 ao modelo, enviamos [0,0,1,...]

In [2]:
def peaks_to_vector(peaks_str, grid_size=512, two_theta_range=(5, 90)):
    try:
        peaks = ast.literal_eval(peaks_str)
    except:
        return np.zeros(grid_size)
    
    grid = np.linspace(two_theta_range[0], two_theta_range[1], grid_size)
    spectrum = np.zeros(grid_size)
    
    for peak in peaks:
        idx = np.argmin(np.abs(grid - peak))
        spectrum[idx] = 1.0
    
    return spectrum

X = np.stack(df['xray_peaks'].apply(peaks_to_vector).values)
print("Shape X:", X.shape)

Shape X: (91145, 512)


Agora fazemos a divisão treino e teste e geramos os tensores de cada um

In [3]:
le = LabelEncoder()
y = le.fit_transform(df['crystal_system'])
print("Classes:", le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

X_tr = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
X_te = torch.tensor(X_test,  dtype=torch.float32).unsqueeze(-1)
y_tr = torch.tensor(y_train, dtype=torch.long)
y_te = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

Classes: ['cubic' 'hexagonal' 'monoclinic' 'orthorhombic' 'tetragonal' 'triclinic'
 'trigonal']
Treino: (72916, 512), Teste: (18229, 512)


Definimos o nosso modelo de forma similar aos MLPs clássicos

In [4]:
class XRDTransformerOriginal(nn.Module):
    def __init__(self, seq_len=512, d_model=512, nhead=8,
                 num_layers=6, dim_feedforward=2048,
                 dropout=0.1, num_classes=7):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(1, d_model)

        #Aqui coloco um positional encoding baseado no que é feito no artigo por didática, mas esse é pouco 
        #usado atualmente. A maioria são PEs treináveis.
        pe = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

        #O dropout zera alguns elementos do tensor aleatoriamente para dificultar overfit.
        self.dropout = nn.Dropout(dropout)

        #Encoder possível para o problema. É usado o clássico do Pythorch
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.dropout(x + self.pe[:, :x.size(1)])
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

Colocamos nosso modelo na GPU e definimos o otimizador e a loss.

In [5]:
model = XRDTransformerOriginal().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

Agora o treino de 100 épocas

In [6]:
for epoch in range(100):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss/len(train_loader):.4f}")

Epoch 01 | Loss: 1.7985


Epoch 02 | Loss: 1.7784


Epoch 03 | Loss: 1.7760


Epoch 04 | Loss: 1.7751


Epoch 05 | Loss: 1.7749


...

Epoch 96 | Loss: 1.7730


Epoch 97 | Loss: 1.7731


Epoch 98 | Loss: 1.7730


Epoch 99 | Loss: 1.7731


Epoch 100 | Loss: 1.7730


Por fim, avaliando o modelo final

In [7]:
model.eval()
all_preds = []

test_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=64, shuffle=False)

with torch.no_grad():
    for xb, _ in test_loader:
        preds_batch = model(xb.to(device)).argmax(dim=1).cpu()
        all_preds.append(preds_batch)

preds = torch.cat(all_preds)
acc = (preds == y_te).float().mean()
print(f"\nAcurácia no teste: {acc:.4f}")


Acurácia no teste: 0.3001


Esse modelo tenta seguir os ensinamentos de Attention is All you Need, limitando o pontencial dos transformers, já que algumas implementações modernas não mais usam Positional Encoding fixo, por exemplo. Dessa forma, a baixa acurácia não reflete um baixo poder do Transformer, mas especificamente desse feito para ser didático e condizente com o que anteriormente proposto.

Com o objetivo de explorara o verdadeiro potencial dele, é treinado um Transformer que explora seu potencial usando funções mais consolidadas no Pythorch. Ele é treinado e discutido em "Transformers_treino".

https://www.kaggle.com/datasets/alessiocorrado99/animals10/data

https://arxiv.org/pdf/1706.03762

https://medium.com/@shravankoninti/history-of-attention-mechanism-an-introduction-to-self-attention-with-visuals-code-explained-a1529c79923e

https://arxiv.org/pdf/1409.0473

https://medium.com/@louiserigny/a-guide-to-understanding-positional-encoding-for-deep-learning-models-fdea4ee938f3

https://arxiv.org/pdf/1810.04805

https://ravjot03.medium.com/bert-vs-gpt-a-guide-to-two-powerful-language-models-b14502438065

